# 05 — Softmax and negative log-likelihood

Part of the micrograd repetition pack.


## Goal
Use the scalar autodiff engine to build class probabilities and a classification loss.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            self.grad += cos(self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        e2x = (2 * self).exp()
        return (e2x - 1) / (e2x + 1)

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __rtruediv__(self, other): return Value(other) * self ** -1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


### Round A — Softmax
Exponentiate each logit, sum the counts, and normalize. Do not use NumPy or PyTorch.


In [ ]:
def softmax(logits):
    """TODO: return one Value probability per logit."""
    raise NotImplementedError

_results.clear()
try:
    probs=softmax([Value(0),Value(3),Value(-2),Value(1)])
    check("four probabilities",len(probs)==4)
    check("probabilities sum to one",close(sum(p.data for p in probs),1))
    check("largest logit has largest probability",max(range(4),key=lambda i:probs[i].data)==1)
except Exception as e: check("softmax runs",False,repr(e))
summary()


### Round B — NLL
Return `-log(probability of the target class)`.


In [ ]:
def nll_loss(logits, target):
    """TODO."""
    raise NotImplementedError

_results.clear()
try:
    logits=[Value(0),Value(3),Value(-2),Value(1)]
    loss=nll_loss(logits,3)
    check("source-notebook loss",close(loss.data,2.1755153626167147))
    loss.backward()
    expected=[0.041772570515350445,0.8390245074625319,0.005653302662216329,-0.8864503806400986]
    for i,e in enumerate(expected):check(f"logit gradient {i}",close(logits[i].grad,e))
    check("logit gradients sum to zero",close(sum(x.grad for x in logits),0))
except Exception as e: check("NLL runs",False,repr(e))
summary()


### Round C — Invariance and stability
Verify that adding the same number to all logits does not change probabilities. Stretch: make a numerically stable softmax by subtracting the maximum logit value.


In [ ]:
def stable_softmax(logits):
    """TODO stretch: avoid overflow while preserving gradients."""
    raise NotImplementedError

_results.clear()
try:
    p1=stable_softmax([Value(0),Value(3),Value(-2),Value(1)])
    p2=stable_softmax([Value(1000),Value(1003),Value(998),Value(1001)])
    check("shift invariance",all(close(a.data,b.data) for a,b in zip(p1,p2)))
    check("large logits remain finite",all(math.isfinite(p.data) for p in p2))
except Exception as e: check("stable softmax",False,repr(e))
summary()


### Concept checks

1. Why do softmax probabilities sum to 1?
2. Why is the correct-class logit gradient `probability - 1`?
3. Why do all logit gradients sum to 0?
4. Repeat with each possible target class.
